# PhaseNet 微调 · 新西兰 GeoNet 数据 (Google Colab)

整条链路：**挂载 Drive → 拉仓库 → 装依赖 → GeoNet FDSN 取数(可续) → 切训练/留出集 → 微调(前后各评合成+真实) → 权重推回 Gitee**。

## Colab 免费版会吃掉成果的两个机制（务必先懂）
1. **断连清空运行时**：`/content/` 下的东西一断连就没了。所以数据和权重全部落到 **Google Drive**（`/content/drive/MyDrive/...`），断连重连数据还在。
2. **单次会话有时限**：取数用 `--max_minutes` 软时限，接近上限主动落盘退出；重连后**重跑同一格**即可从断点续拉、续训。

> 用法：从上到下依次运行。任何一格断了，重连后从该格重跑，不会从头再来。

## 0. 检查 GPU（Runtime → Change runtime type → GPU）

In [ ]:
!nvidia-smi -L || echo '没分到 GPU：菜单 Runtime → Change runtime type → 选 GPU，再重连'

## 1. 挂载 Google Drive（数据/权重的永久家，断连不丢）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 所有产物都放这个 Drive 目录，断连/重连都在
WORK = '/content/drive/MyDrive/dizheng'
DATA_DIR = os.path.join(WORK, 'data')
RUN_DIR  = os.path.join(WORK, 'runs', 'geonet_ft1')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
print('产物目录(Drive):', WORK)

## 2. 拉仓库（GitHub 是最新版）
已存在就 `git pull`，避免重复 clone。

In [ ]:
import os
REPO = '/content/dz'
GITHUB_URL = 'https://github.com/ixijxjgxidj-cmd/dz.git'
if os.path.isdir(os.path.join(REPO, '.git')):
    !cd {REPO} && git pull --ff-only
else:
    !git clone {GITHUB_URL} {REPO}
%cd {REPO}
!git log --oneline -3

## 3. 装依赖
Colab 自带 torch（别动，动了容易 CUDA 版本冲突）。只补 seisbench / obspy / h5py。

In [ ]:
!pip -q install seisbench obspy h5py
import torch, seisbench, obspy, h5py
print('torch', torch.__version__, '| CUDA可用', torch.cuda.is_available())
print('seisbench', seisbench.__version__, '| obspy', obspy.__version__)

## 4. GeoNet 取数（FDSN 直连，可断点续传）

输出直接写到 **Drive**。参数含义：
- `--start/--end`：事件时间窗（先小后大，跑通再放量）
- `--minmag/--maxmag`：震级范围（3~6 波形信噪比好、又避开超大震削波）
- `--max_minutes`：软时限，接近 Colab 会话上限前主动落盘退出
- `--max_gb/--max_traces`：子集大小上限

**断连了怎么办**：重连后重跑本格，`progress.json` 会让它从上次的事件继续，不重复下载。

In [ ]:
GEONET_H5 = os.path.join(DATA_DIR, 'geonet_100hz.hdf5')

# 先用小窗口跑通（1 个月、时限 20 分钟）。跑通后把 --start/--end 拉长、--max_minutes 调大再放量。
!python scripts/geonet_fetch.py \
  --out {GEONET_H5} \
  --start 2023-01-01 --end 2023-02-01 \
  --minmag 3.0 --maxmag 6.0 \
  --maxstations 8 --max_traces 20000 --max_gb 8 \
  --max_minutes 20 --save_every 20

## 5. 切训练集 / 留出集
留出集是**训练不碰**的真波形，微调后在它上面评分，才是泛化的真证据。
按 key 排序后取后 15% 作留出（固定切法，可复现）。

In [ ]:
import h5py, os

TRAIN_H5   = os.path.join(DATA_DIR, 'geonet_train.hdf5')
HOLDOUT_H5 = os.path.join(DATA_DIR, 'geonet_holdout.hdf5')
HOLDOUT_FRAC = 0.15

with h5py.File(GEONET_H5, 'r') as f:
    keys = sorted(f['data'].keys())
n = len(keys)
assert n > 0, '取数结果为空：把第 4 格的时间窗放大 / 时限调长后重跑'
n_hold = max(1, int(round(n * HOLDOUT_FRAC)))
hold_keys = set(keys[-n_hold:])
print(f'总 {n} 条 -> 训练 {n - n_hold} 条 / 留出 {n_hold} 条')

def copy_subset(src, dst, want_keys):
    with h5py.File(src, 'r') as fin, h5py.File(dst, 'w') as fout:
        g_in = fin['data']; g_out = fout.create_group('data')
        for k in g_in:
            if k in want_keys:
                d = g_out.create_dataset(k, data=g_in[k][...], compression='gzip')
                for ak, av in g_in[k].attrs.items():
                    d.attrs[ak] = av

copy_subset(GEONET_H5, TRAIN_H5,   set(keys[:-n_hold]))
copy_subset(GEONET_H5, HOLDOUT_H5, hold_keys)
print('训练集:', TRAIN_H5)
print('留出集:', HOLDOUT_H5)

## 6. 微调（前后各评一次：合成 + 真实留出）

- `--data`：训练集（真波形）
- `--holdout`：留出集，微调前后各评一次，直接看真实数据上有没有提升
- `--out` 指到 **Drive**，`best.pt`/`last.pt` 断连不丢
- 断连后重跑本格加 `--resume` 即可续训

BN 默认冻结（防小样本冲坏 running stats，这是之前崩溃的根因），lr 小、grad-clip 开。

In [ ]:
# 首次训练（不加 --resume）。断连后重连、想接着练：在末尾加 --resume 重跑本格。
!python scripts/finetune_phasenet.py \
  --data {TRAIN_H5} \
  --holdout {HOLDOUT_H5} --holdout-max 200 \
  --out {RUN_DIR} \
  --epochs 10 --batch 16 --lr 3e-5 --grad-clip 1.0 --score-every 1

## 7.（断连恢复）续训
会话被回收后：重跑第 1~3 格挂载/拉仓库/装依赖，然后跑这一格续训（数据和 checkpoint 都在 Drive）。

In [ ]:
!python scripts/finetune_phasenet.py \
  --data {TRAIN_H5} \
  --holdout {HOLDOUT_H5} --holdout-max 200 \
  --out {RUN_DIR} \
  --epochs 10 --batch 16 --lr 3e-5 --grad-clip 1.0 --score-every 1 \
  --resume

## 8. 保住成果：把权重推回 Gitee
`best.pt` 只有几 MB。先复制进仓库 `weights/`，再提交推送。

> **注意**：下面的推送需要你的 Gitee 账号密码/令牌，会真的改动远程仓库。确认无误再运行。

In [ ]:
import shutil, os
src = os.path.join(RUN_DIR, 'best.pt')
dst = os.path.join(REPO, 'weights', 'phasenet_geonet_ft1_best.pt')
shutil.copy(src, dst)
print('已复制:', dst, '(', round(os.path.getsize(dst)/1e6, 2), 'MB )')

# 也把留出集评分快照留一份，方便回看这次微调到底有没有用
prog = os.path.join(RUN_DIR, 'progress.json')
if os.path.exists(prog):
    shutil.copy(prog, os.path.join(REPO, 'weights', 'phasenet_geonet_ft1_progress.json'))
    print('已复制评分快照 progress.json')

In [ ]:
# 填你自己的信息（令牌不要写进会被提交的文件）
GIT_NAME  = 'hulk-cheng'
GIT_EMAIL = 'you@example.com'
!cd {REPO} && git config user.name  "{GIT_NAME}"
!cd {REPO} && git config user.email "{GIT_EMAIL}"
!cd {REPO} && git add weights/phasenet_geonet_ft1_best.pt weights/phasenet_geonet_ft1_progress.json
!cd {REPO} && git commit -m "新增 GeoNet(新西兰) 微调权重 + 留出集评分"
print('已本地提交。下一格推送到 Gitee。')

In [ ]:
# 推送到 Gitee。运行时会让你输入用户名 + 令牌(或密码)。
# Gitee 备份地址；如果你只想推 GitHub，把 URL 换成 github 那个。
GITEE_URL = 'https://gitee.com/hulk-cheng/dizheng.git'
!cd {REPO} && git push {GITEE_URL} HEAD:main

## 怎么判断这次微调成功了
看第 6/7 格结尾的 **对比** 输出：

- **[合成]** 微调前后都应接近 `2.0`。关键是**微调后没掉到 0**——证明没重蹈 BN 崩溃。
- **[真实]** 才是泛化的真证据：
  - 微调前(真实) 分数 = 预训练 stead 模型在新西兰数据上的水平；
  - 微调后(真实) 分数更高 = 这次微调**真的学到了**新西兰数据的特性；
  - 若微调后(真实)反而更低 = 过拟合或学习率过大，调小 `--lr`、减小 `--epochs` 或增大留出集重来。

换日本数据同理：日本(Hi-net/K-NET)需注册下载，拿到后转成同格式 HDF5（group `data` + attrs `p_sample_100hz`/`s_sample_100hz`）即可复用第 5~7 格。